In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import numpy as np

from datetime import datetime

In [3]:
current_time = datetime.now().strftime("%Y%m%d%H%M%S")

In [4]:
progress_file = f'/group/pmc021/amunif/epi-thesis/workflow/04. Neural Network/progress_conv1d_{current_time}.txt'

In [5]:
dataset_path = "/group/pmc021/amunif/epi-thesis/dataset/"

def load_large_csv(file_name, chunksize=20000):
    # Read the CSV file
    mylist = []

    for chunk in pd.read_csv(file_name, chunksize = chunksize):
        mylist.append(chunk)

    df = pd.concat(mylist, axis = 0)
    
    del mylist
    return df

In [6]:
def save_progress(file_name, message):
    with open(file_name, 'a+') as file:
        file.write(message + "\n")

In [9]:
# Load the data
X = pd.read_csv(f"{dataset_path}histone_features.csv", nrows=100)
y = pd.read_csv(f"{dataset_path}value_1_df.csv", nrows=100)

In [10]:
# Convert to numpy
X_np = X.to_numpy()
y_np = y.to_numpy()

In [13]:
X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(X_np, y_np, test_size=0.2, random_state=42)

In [23]:
# Convert NumPy arrays to PyTorch tensors
X_train_tensor = torch.tensor(X_train_np, dtype=torch.float32).unsqueeze(1)
y_train_tensor = torch.tensor(y_train_np, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_np, dtype=torch.float32).unsqueeze(1)
y_test_tensor = torch.tensor(y_test_np, dtype=torch.float32)

In [24]:
# Create TensorDataset for training and testing sets
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [25]:
# Create a DataLoader
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [26]:
# Create CNN1D class
class HighDimCNN1D(nn.Module):
    def __init__(self, input_size):
        super(HighDimCNN1D, self).__init__()
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool1d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1)
        self.pool2 = nn.MaxPool1d(kernel_size=2, stride=2)
        self.conv3 = nn.Conv1d(in_channels=128, out_channels=256, kernel_size=3, stride=1, padding=1)
        self.pool3 = nn.MaxPool1d(kernel_size=2, stride=2)
        
        # Calculate the size after the convolution and pooling layers
        conv_output_size = input_size // 8  # Adjust this based on the number of pooling layers
        
        self.fc1 = nn.Linear(256 * conv_output_size, 512)
        self.fc2 = nn.Linear(512, 1)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool2(torch.relu(self.conv2(x)))
        x = self.pool3(torch.relu(self.conv3(x)))
        x = x.view(x.size(0), -1)  # Flatten the tensor
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [27]:
input_size = 20000  # Number of features in the input
model = HighDimCNN1D(input_size=input_size)

In [28]:
# Loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [29]:
num_epochs = 100

In [30]:
# Training loop
for epoch in range(num_epochs): 
    model.train()
    for inputs, targets in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
    print(f'Epoch {epoch+1}, Loss: {loss.item()}')

Epoch 1, Loss: 3371.89013671875
Epoch 2, Loss: 221.79586791992188
Epoch 3, Loss: 80.43352508544922
Epoch 4, Loss: 9.281341552734375
Epoch 5, Loss: 41.95248794555664
Epoch 6, Loss: 130.55447387695312
Epoch 7, Loss: 87.05644226074219
Epoch 8, Loss: 103.64060974121094
Epoch 9, Loss: 37.714454650878906
Epoch 10, Loss: 418.00994873046875
Epoch 11, Loss: 2599.977783203125
Epoch 12, Loss: 2221.697021484375
Epoch 13, Loss: 203.96043395996094
Epoch 14, Loss: 354.12738037109375
Epoch 15, Loss: 83.3335952758789
Epoch 16, Loss: 2385.476318359375
Epoch 17, Loss: 78.43090057373047
Epoch 18, Loss: 2365.87744140625
Epoch 19, Loss: 2257.051025390625
Epoch 20, Loss: 125.2521743774414
Epoch 21, Loss: 327.6352233886719
Epoch 22, Loss: 2252.83349609375
Epoch 23, Loss: 55.99394226074219
Epoch 24, Loss: 37.27030563354492
Epoch 25, Loss: 2289.05810546875
Epoch 26, Loss: 2795.361572265625
Epoch 27, Loss: 95.52123260498047
Epoch 28, Loss: 314.8896179199219
Epoch 29, Loss: 304.8822326660156
Epoch 30, Loss: 2668.

In [31]:
# Evaluation
model.eval()
all_targets = []
all_predictions = []

In [32]:
with torch.no_grad():
    for inputs, targets in test_loader:
        outputs = model(inputs)
        all_targets.extend(targets.numpy())
        all_predictions.extend(outputs.numpy())

In [33]:
all_targets = np.array(all_targets)
all_predictions = np.array(all_predictions)

mse = mean_squared_error(all_targets, all_predictions)
rmse = np.sqrt(mse)
mae = mean_absolute_error(all_targets, all_predictions)
r2 = r2_score(all_targets, all_predictions)

print(f'MSE: {mse}')
print(f'RMSE: {rmse}')
print(f'MAE: {mae}')
print(f'R2 Score: {r2}')

MSE: 0.34687119722366333
RMSE: 0.5889577269554138
MAE: 0.3921221196651459
R2 Score: 0.9998082518577576
